In [0]:
from pyspark.sql import functions as F

bronze_df = spark.table("bronze.world_bank_food_prices")

print("Bronze rows:", bronze_df.count())

In [0]:
required_columns = [
    "ISO3",
    "country",
    "geo_id",
    "price_date",
    "lat",
    "lon"
]

missing_columns = [
    column for column in required_columns
    if column not in bronze_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

In [0]:
# ---------------------------------------------------------
# CLEAN STRINGS
# ---------------------------------------------------------

clean_df = bronze_df

for column in bronze_df.columns:
    if column not in [
        "_ingestion_timestamp",
        "load_timestamp"
    ]:
        clean_df = clean_df.withColumn(
            column,
            F.when(
                F.trim(F.col(column).cast("string")) == "",
                F.lit(None)
            ).otherwise(
                F.trim(F.col(column).cast("string"))
            )
        )

# ---------------------------------------------------------
# CONVERT DATA TYPES
# ---------------------------------------------------------

clean_df = (
    clean_df
    .withColumn("ISO3", F.upper(F.col("ISO3")))
    .withColumn(
        "_price_date",
        F.expr("try_cast(price_date AS DATE)")
    )
    .withColumn(
        "_lat",
        F.expr("try_cast(lat AS DOUBLE)")
    )
    .withColumn(
        "_lon",
        F.expr("try_cast(lon AS DOUBLE)")
    )
)

# Convert year and month only if those columns exist
if "year" in clean_df.columns:
    clean_df = clean_df.withColumn(
        "_year",
        F.expr("try_cast(year AS INT)")
    )

if "month" in clean_df.columns:
    clean_df = clean_df.withColumn(
        "_month",
        F.expr("try_cast(month AS INT)")
    )

# Validate the price column if present
price_column = (
    "mp_price" if "mp_price" in clean_df.columns
    else "price" if "price" in clean_df.columns
    else None
)

if price_column:
    clean_df = clean_df.withColumn(
        "_numeric_price",
        F.expr(
            f"try_cast(`{price_column}` AS DOUBLE)"
        )
    )

# ---------------------------------------------------------
# BUILD VALIDATION ERRORS
# ---------------------------------------------------------

errors = [
    F.when(
        F.col("ISO3").isNull(),
        F.lit("Missing ISO3")
    ),
    F.when(
        F.col("country").isNull(),
        F.lit("Missing country")
    ),
    F.when(
        F.col("geo_id").isNull(),
        F.lit("Missing geo_id")
    ),
    F.when(
        F.col("_price_date").isNull(),
        F.lit("Invalid price_date")
    ),
    F.when(
        F.col("_lat").isNull() |
        (F.col("_lat") < -90) |
        (F.col("_lat") > 90),
        F.lit("Invalid latitude")
    ),
    F.when(
        F.col("_lon").isNull() |
        (F.col("_lon") < -180) |
        (F.col("_lon") > 180),
        F.lit("Invalid longitude")
    )
]

# Check that ISO3 has the expected three-character format
errors.append(
    F.when(
        F.length(F.col("ISO3")) != 3,
        F.lit("Invalid ISO3 length")
    )
)

# Check price conversion only when a price field exists
if price_column:
    errors.append(
        F.when(
            F.col(price_column).isNotNull() &
            F.col("_numeric_price").isNull(),
            F.lit("Invalid numeric price")
        )
    )

# Check year and month conversion where applicable
if "year" in clean_df.columns:
    errors.append(
        F.when(
            F.col("year").isNotNull() &
            F.col("_year").isNull(),
            F.lit("Invalid year")
        )
    )

if "month" in clean_df.columns:
    errors.append(
        F.when(
            F.col("month").isNotNull() &
            F.col("_month").isNull(),
            F.lit("Invalid month")
        )
    )

clean_df = clean_df.withColumn(
    "_validation_error",
    F.concat_ws(
        "; ",
        *[error.otherwise(None) for error in errors]
    )
)

valid_df = clean_df.filter(
    F.col("_validation_error") == ""
)

reject_df = clean_df.filter(
    F.col("_validation_error") != ""
)

print("Bronze records:", bronze_df.count())
print("Valid records:", valid_df.count())
print("Rejected records:", reject_df.count())

In [0]:

# =========================================================
# VALIDATION AUDIT + QUARANTINE REJECTED RECORDS
# =========================================================

import uuid
from datetime import datetime, timezone
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    LongType
)

# ---------------------------------------------------------
# 1. PREPARE AUDIT TABLE
# ---------------------------------------------------------

spark.sql("CREATE SCHEMA IF NOT EXISTS audit")
spark.sql("CREATE SCHEMA IF NOT EXISTS rejects")

audit_table = "audit.pipeline_runs"
reject_table = "rejects.bronze_validation_rejects"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {audit_table} (
    run_id STRING,
    source_name STRING,
    layer STRING,
    load_type STRING,
    parameter_processed STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    records_read BIGINT,
    records_inserted BIGINT,
    records_updated BIGINT,
    records_rejected BIGINT,
    status STRING,
    error_message STRING
) USING DELTA
""")

# Add this column only if the existing audit table lacks it.
if "parameter_processed" not in spark.table(audit_table).columns:
    spark.sql(
        f"ALTER TABLE {audit_table} "
        "ADD COLUMNS (parameter_processed STRING)"
    )

# ---------------------------------------------------------
# 2. CREATE A UNIQUE VALIDATION RUN ID AND COUNTS
# ---------------------------------------------------------

validation_run_id = str(uuid.uuid4())
validation_start = datetime.now(timezone.utc).replace(tzinfo=None)

validation_rows_read = bronze_df.count()
validation_rows_valid = valid_df.count()
validation_rows_rejected = reject_df.count()

print("Validation run ID:", validation_run_id)
print("Bronze records:", validation_rows_read)
print("Valid records:", validation_rows_valid)
print("Rejected records:", validation_rows_rejected)

# ---------------------------------------------------------
# 3. WRITE REJECTED RECORDS TO A SEPARATE QUARANTINE TABLE
# ---------------------------------------------------------

quarantine_df = (
    reject_df
    .withColumn("reject_run_id", F.lit(validation_run_id))
    .withColumn("rejected_at", F.current_timestamp())
)

# Append only when there are rejected rows.
# Never overwrite or drop an existing rejects table.
if validation_rows_rejected > 0:
    (
        quarantine_df.write
        .format("delta")
        .option("mergeSchema", "true")
        .mode("append")
        .saveAsTable(reject_table)
    )

print("Validation quarantine table:", reject_table)
print("Rejected records processed:", validation_rows_rejected)

# ---------------------------------------------------------
# 4. WRITE SUCCESS AUDIT ONLY AFTER QUARANTINE SUCCEEDS
# ---------------------------------------------------------

validation_end = datetime.now(timezone.utc).replace(tzinfo=None)

audit_schema = StructType([
    StructField("run_id", StringType(), False),
    StructField("source_name", StringType(), True),
    StructField("layer", StringType(), True),
    StructField("load_type", StringType(), True),
    StructField("parameter_processed", StringType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("records_read", LongType(), True),
    StructField("records_inserted", LongType(), True),
    StructField("records_updated", LongType(), True),
    StructField("records_rejected", LongType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True),
])

audit_row = spark.createDataFrame(
    [(
        validation_run_id,
        "World Bank Real-Time Food Prices",
        "Validation",
        "validation",
        "table=bronze.world_bank_food_prices",
        validation_start,
        validation_end,
        int(validation_rows_read),
        0,
        0,
        int(validation_rows_rejected),
        "SUCCESS",
        None
    )],
    schema=audit_schema
)

# Align the new row with the existing table's column order.
table_columns = spark.table(audit_table).columns

(
    audit_row.select(*table_columns)
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(audit_table)
)

print("Validation audit record created successfully.")

# ---------------------------------------------------------
# 5. DISPLAY THIS RUN'S AUDIT RECORD AND QUARANTINE COUNT
# ---------------------------------------------------------

display(
    spark.table(audit_table)
    .filter(F.col("run_id") == validation_run_id)
)

if spark.catalog.tableExists(reject_table):
    display(
        spark.table(reject_table)
        .filter(F.col("reject_run_id") == validation_run_id)
        .agg(F.count("*").alias("rejected_records_this_run"))
    )
else:
    print("No rejected records; the quarantine table has not been created.")


In [0]:
display(
    spark.table("audit.pipeline_runs")
    .filter(F.col("layer") == "Validation")
    .orderBy(F.col("start_time").desc())
)


In [0]:
# =========================================================
# SILVER: QUARANTINE REJECTED RECORDS
# =========================================================
import uuid
spark.sql("CREATE SCHEMA IF NOT EXISTS rejects")

reject_table = "rejects.silver_food_price_rejects"

silver_run_id = str(uuid.uuid4())

silver_rejects = (
    reject_df
    .withColumn(
        "reject_run_id",
        F.lit(silver_run_id)
    )
    .withColumn(
        "rejected_at",
        F.current_timestamp()
    )
)

rejected_count = silver_rejects.count()

if rejected_count > 0:
    (
        silver_rejects.write
        .format("delta")
        .option("mergeSchema", "true")
        .mode("append")
        .saveAsTable(reject_table)
    )

print("Rejected records:", rejected_count)
print("Reject table:", reject_table)

In [0]:
display(
    spark.sql("""
        select count(*) as rejected_records
        from rejects.silver_food_price_rejects
    """)
)

In [0]:

# DATA DICTIONARY FOR THE README
# Primary key columns are marked PK.

def dictionary(table, pk):
    print(f"\n### {table}\n")
    print("| Column | Data type | Primary key |")
    print("|---|---|---|")

    for field in spark.table(table).schema.fields:
        primary_key = "PK" if field.name in pk else ""
        print(
            f"| {field.name} | "
            f"{field.dataType.simpleString()} | "
            f"{primary_key} |"
        )

tables = [
    ("bronze.world_bank_food_prices", []),
    ("silver.world_bank_food_prices", ["ISO3", "geo_id", "price_date"]),
    ("rejects.silver_food_price_rejects_v2", []),
    ("audit.pipeline_runs", ["run_id", "layer"]),
]

for table, primary_keys in tables:
    if spark.catalog.tableExists(table):
        dictionary(table, primary_keys)
    else:
        print(f"\nSkipping missing table: {table}")
